# Opentrons Flex

The Opentrons Flex is a liquid handler with up to two independent pipette mounts (1-, 8-, or 96-channel), an optional gripper, and up to four temperature/heater-shaker/thermocycler modules. PyLabRobot talks to it over the robot's built-in HTTP robot-server API — the same server the Opentrons App uses — rather than through the Python Protocol API.

| Property | Value |
|---|---|
| [OEM link](https://opentrons.com/flex) | |
| [HTTP API reference](https://docs.opentrons.com/v2/) | |
| Communication | HTTP, robot-server API, default port 31950 |
| PLR class | `OpentronsFlex` |
| PLR base class | `OpentronsRobot` (shared with a future OT-2 implementation) |

```{warning}
This driver has NOT been tested against real hardware in PyLabRobot. The command
payloads follow the Opentrons HTTP robot-server API as documented and observed
on other Flex integrations. If you verify it on a Flex, please open a PR to
remove this warning.
```

## How it talks

`OpentronsFlex` (and its shared base, `OpentronsRobot`) speaks HTTP to the robot-server running on the Flex itself, not the Python Protocol API. `setup()`:

1. Opens an `httpx.AsyncClient` and checks `GET /health` to confirm the robot-server (not Jupyter/Python API mode) is reachable.
2. Creates an empty run with `POST /runs` — an empty run (no `protocolId`) lets PLR send interactive setup commands instead of a pre-baked protocol.
3. Discovers the mounted pipette via `GET /instruments` and loads it into the run (`loadPipette`).
4. Homes all axes.

Every subsequent operation (`pickUpTip`, `aspirate`, `dispense`, `loadLabware`, ...) is POSTed as a command to that run (`POST /runs/{run_id}/commands`) and polled via `GET` until it succeeds or fails.

## Physical setup / cabling

Connect to the Flex over the network:

- **Ethernet (recommended):** connect the Flex's ethernet port to the same network as your computer, or directly via a USB-C-to-ethernet adapter. Find the robot's IP address on its touchscreen under *Settings → Network*.
- **Wi-Fi:** configure Wi-Fi on the touchscreen; the IP address is shown in the same network settings screen.

The robot-server listens on port `31950` by default. No USB driver or vendor SDK installation is required on the computer side — just `pip install pylabrobot[http]` (or ensure `httpx` is installed).

## Setup

Construct `OpentronsFlex` with a `FlexDeck` and the robot's IP address. `setup()` connects, creates a run, and discovers the mounted pipette (`flex.pipette`).

In [ ]:
from pylabrobot.opentrons import OpentronsFlex
from pylabrobot.resources.opentrons import FlexDeck

flex = OpentronsFlex(deck=FlexDeck(), host="169.254.99.87")  # replace with your Flex's IP
await flex.setup()

print("Pipette:", flex.pipette.pipette_name, f"({flex.pipette.channels} channels)")

## Deck layout

Load a Flex 50 uL tip rack and a Corning 96-well plate onto the deck. Corning labware isn't in the built-in Flex load-name map, so set `ot_load_name` explicitly to the Opentrons labware-library name.

In [ ]:
from pylabrobot.resources.corning import Cor_96_wellplate_360ul_Fb
from pylabrobot.resources.opentrons import flex_96_tiprack_50ul

tiprack = flex_96_tiprack_50ul(name="tips_01")
plate = Cor_96_wellplate_360ul_Fb(name="plate_01")
plate.ot_load_name = "corning_96_wellplate_360ul_flat"

flex.deck.assign_child_at_slot(tiprack, slot="C1")
flex.deck.assign_child_at_slot(plate, slot="D1")

## Picking up tips

`pick_up_tips` takes a list of `TipSpot`s and a matching `use_channels` list — here, channel 0 picks up the tip at well A1 of the rack. The op commits directly to the tip spot's tracker on the resource tree (`tiprack["A1"]` now reports no tip).

In [ ]:
await flex.pick_up_tips(tiprack["A1"], use_channels=[0])

## Aspirating

The resource tree's volume tracker needs to know a well has liquid before you can aspirate from it — this driver has no liquid-level probe yet, so set the volume manually. Then aspirate 20 uL from well A1 of the plate on channel 0.

In [ ]:
plate["A1"][0].tracker.set_volume(200.0)
plate["A1"][0].tracker.commit()

await flex.aspirate(plate["A1"], vols=[20.0], use_channels=[0])

## Dispensing

Dispense the 20 uL into well A2 of the same plate, again on channel 0.

In [ ]:
await flex.dispense(plate["A2"], vols=[20.0], use_channels=[0])

## Dropping tips

Return the tip to its original spot in the rack. Dropping into a `Trash` resource instead (e.g. `flex.deck.get_trash_area()`) routes through the Flex's addressable-area drop-tip commands and does not restore the tracker.

In [ ]:
await flex.drop_tips(tiprack["A1"], use_channels=[0])

## Teardown

In [ ]:
await flex.stop()